### **Implementation of Communication-Efficient Federated Learning using Model Update Optimization**

* **Introduction:** Federated Learning involves frequent communication between clients and server, which can lead to high communication overhead. This assignment focuses on reducing communication cost while maintaining model performance.

* **Methodology:** Clients train local models and send optimized or compressed updates to the server instead of full model weights. The server aggregates these updates using the Federated Averaging algorithm.

* **Working:** Each client trains locally
Model updates are compressed or optimized
Reduced-size updates are sent to the server
Server aggregates updates
Global model is updated and redistributed

* **Result:** The system achieves faster communication and reduced data transfer while maintaining acceptable model accuracy.

* **Conclusion:** This assignment demonstrates that communication efficiency is crucial in federated learning systems. Optimizing model updates can significantly improve scalability without major performance loss.

In [2]:
# ============================================================
# Vertical Federated Learning (VFL) on Healthcare Dataset
# Includes: Clean Structure + Accuracy + Better Logging
# ============================================================

import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.datasets import load_breast_cancer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

# ------------------------------------------------------------
# 1. Load and Preprocess Healthcare Dataset
# ------------------------------------------------------------
data = load_breast_cancer()

X = data.data
y = data.target

# Normalize features
scaler = StandardScaler()
X = scaler.fit_transform(X)

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Convert to tensors
X_train = torch.tensor(X_train, dtype=torch.float32)
X_test = torch.tensor(X_test, dtype=torch.float32)

y_train = torch.tensor(y_train, dtype=torch.float32).view(-1, 1)
y_test = torch.tensor(y_test, dtype=torch.float32).view(-1, 1)

# ------------------------------------------------------------
# 2. Vertical Data Partition (Feature Split Across Clients)
# ------------------------------------------------------------
# Client A gets first half features
XA_train = X_train[:, :15]
XA_test = X_test[:, :15]

# Client B gets remaining features
XB_train = X_train[:, 15:]
XB_test = X_test[:, 15:]

# ------------------------------------------------------------
# 3. Define Client Models
# ------------------------------------------------------------
class ClientModel(nn.Module):
    """
    Each client learns feature representation (embedding)
    from its own subset of features.
    """
    def __init__(self, input_size):
        super(ClientModel, self).__init__()
        self.fc = nn.Linear(input_size, 8)

    def forward(self, x):
        return torch.relu(self.fc(x))


# ------------------------------------------------------------
# 4. Define Server Model
# ------------------------------------------------------------
class ServerModel(nn.Module):
    """
    Server combines embeddings from all clients
    and performs final prediction.
    """
    def __init__(self):
        super(ServerModel, self).__init__()
        self.fc = nn.Linear(16, 1)

    def forward(self, xa, xb):
        combined = torch.cat((xa, xb), dim=1)
        return torch.sigmoid(self.fc(combined))


# ------------------------------------------------------------
# 5. Initialize Models and Optimizers
# ------------------------------------------------------------
clientA = ClientModel(15)
clientB = ClientModel(XB_train.shape[1])
server = ServerModel()

criterion = nn.BCELoss()

optA = optim.Adam(clientA.parameters(), lr=0.001)
optB = optim.Adam(clientB.parameters(), lr=0.001)
optS = optim.Adam(server.parameters(), lr=0.001)

# ------------------------------------------------------------
# 6. Training Loop
# ------------------------------------------------------------
epochs = 50
loss_history = []

print("\n========== Training Started (Vertical FL) ==========\n")

for epoch in range(epochs):

    # Step 1: Clients generate embeddings
    embedA = clientA(XA_train)
    embedB = clientB(XB_train)

    # Step 2: Server makes prediction
    preds = server(embedA, embedB)

    # Step 3: Compute loss
    loss = criterion(preds, y_train)
    loss_history.append(loss.item())

    # Step 4: Backpropagation across all models
    optA.zero_grad()
    optB.zero_grad()
    optS.zero_grad()

    loss.backward()

    optA.step()
    optB.step()
    optS.step()

    # Logging every 10 epochs
    if (epoch + 1) % 10 == 0:
        print(f"Epoch [{epoch+1}/{epochs}] - Loss: {loss.item():.4f}")

print("\n========== Training Completed ==========\n")

# ------------------------------------------------------------
# 7. Evaluation Function
# ------------------------------------------------------------
def evaluate():
    """
    Evaluates model on test data
    """
    clientA.eval()
    clientB.eval()
    server.eval()

    with torch.no_grad():
        embedA = clientA(XA_test)
        embedB = clientB(XB_test)

        preds = server(embedA, embedB)
        predicted = (preds > 0.5).float()

        accuracy = (predicted == y_test).float().mean()

    return accuracy.item()


# ------------------------------------------------------------
# 8. Run Evaluation
# ------------------------------------------------------------
accuracy = evaluate()

print(f"Test Accuracy: {accuracy * 100:.2f}%")


========== Training Started (Vertical FL) ==========

Epoch [10/50] - Loss: 0.6295
Epoch [20/50] - Loss: 0.5839
Epoch [30/50] - Loss: 0.5415
Epoch [40/50] - Loss: 0.5013
Epoch [50/50] - Loss: 0.4628

========== Training Completed ==========

Test Accuracy: 86.84%
